**Merge Google Drive Batches**

This notebook merges the three batches in which Google Drive downloaded the gravestone photos into one folder per section of the cemetery. Each batch in `./data/raw` repeats the same folder tree (`<section>/Row N/Stone M`) but holds a different subset of the photos, so no single batch is complete. It outputs exclusively files: every photo is copied to the same path inside `./data`, the raw batches are read but never modified, and an existing file is never overwritten, so the notebook can be re-run safely:

1. `./data/Women`: every photo of the women's section, one folder per row and stone
2. `./data/Children`: every photo of the children's section, laid out the same way

**1. Configuration and Helpers**

This cell defines the paths and functions that every later cell reuses:

1. `RAW_DIR` and `OUTPUT_DIR`: the batches are read from `./data/raw`, and the merged sections are written to `./data`
2. `list_photos`: every photo of a folder as a path relative to that folder, leaving out system files such as `.DS_Store`. A photo is identified by this relative path, e.g. `Women/Row 1/Stone 2/front.jpg`
3. `merge_batches`: the merge rule. A photo missing from the output is copied, a photo already there with identical bytes is skipped, and a photo already there with different bytes is reported as a conflict, keeping the first batch's file

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and helpers  —  shared by every later cell
#
# The three Google Drive batches repeat one folder tree,
#     <batch>/<section>/Row N/Stone M/<view>.<ext>
# and each holds a different subset of the photos. A photo is identified by
# its path relative to its batch, so the merge is a copy keyed on that path:
#   • absent from the output         → copied
#   • present with identical bytes   → skipped (a duplicate, or an earlier run)
#   • present with different bytes   → conflict: the first batch's file is
#                                      kept, and nothing is ever overwritten
# Batches are visited in name order, so "first" means the same on every run.
# ─────────────────────────────────────────────────────────────────────────────
import filecmp
import shutil
from dataclasses import dataclass, field
from pathlib import Path

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
DATA_DIR   = Path("data")
RAW_DIR    = DATA_DIR / "raw"     # one subfolder per download batch, read only
OUTPUT_DIR = DATA_DIR             # receives one folder per cemetery section
MAX_LISTED = 10                   # problem paths printed per report line
RULE       = "─" * 72             # frames every printed report

# Files written by the operating system rather than by the fieldwork. They are
# left out of every listing, so they are neither counted nor copied.
IGNORED_FILE_NAMES = {".DS_Store", "Thumbs.db", "desktop.ini"}

# ═════════════════════════════════════════════════════════════════════════════
# FILE LISTING  —  every photo is addressed by its path relative to its batch
# ═════════════════════════════════════════════════════════════════════════════
def find_batch_folders(raw_dir: Path) -> list[Path]:
    """Return the batch folders inside the raw folder.

    Args:
        raw_dir: the folder holding one subfolder per download batch.

    Returns:
        The batch folders, sorted by name so that the merge order is fixed.

    Raises:
        FileNotFoundError: if `raw_dir` does not exist.
    """
    if not raw_dir.is_dir():
        raise FileNotFoundError(f"Raw folder not found: {raw_dir}")
    return sorted(path for path in raw_dir.iterdir() if path.is_dir())


def is_ignored(path: Path) -> bool:
    """Say whether a file is left out of listings and of the merge.

    Args:
        path: the file to check.

    Returns:
        True for operating-system files (such as `.DS_Store`) and hidden
        files, whose names start with a dot; False for photos.
    """
    return path.name in IGNORED_FILE_NAMES or path.name.startswith(".")


def list_photos(folder: Path) -> list[Path]:
    """List every photo inside a folder, relative to that folder.

    Args:
        folder: a batch folder, or a section folder of the merged output.

    Returns:
        Sorted relative paths such as `Women/Row 1/Stone 1/front.jpg`,
        leaving out ignored files.
    """
    return sorted(path.relative_to(folder) for path in folder.rglob("*")
                  if path.is_file() and not is_ignored(path))

# ═════════════════════════════════════════════════════════════════════════════
# MERGE  —  copy keyed on relative path; identical files skipped, others kept
# ═════════════════════════════════════════════════════════════════════════════
@dataclass
class Conflict:
    """Two batches hold different files at the same relative path.

    Attributes:
        relative_path: the path shared by both files, relative to the batch.
        kept: the file in the output, copied from the first batch.
        skipped: the file from a later batch, which was not copied.
    """

    relative_path: Path
    kept: Path
    skipped: Path


@dataclass
class MergeReport:
    """What one merge did, as lists of relative paths.

    Attributes:
        copied: photos copied into the output by this run.
        already_present: photos skipped because an identical file was
            already in the output (a duplicate across batches, or a photo
            copied by an earlier run).
        conflicts: paths where two batches hold different files.
    """

    copied: list[Path] = field(default_factory=list)
    already_present: list[Path] = field(default_factory=list)
    conflicts: list[Conflict] = field(default_factory=list)


def copy_photo(source: Path, destination: Path) -> None:
    """Copy one photo, creating its row and stone folders when needed.

    `shutil.copy2` also keeps the file's modification time, which records
    when the photo was taken or downloaded.

    Args:
        source: the photo inside a batch.
        destination: where the copy goes inside the output.
    """
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)


def merge_batches(raw_dir: Path, output_dir: Path) -> MergeReport:
    """Copy the photos of every batch into one output tree.

    Args:
        raw_dir: the folder holding one subfolder per download batch.
        output_dir: the folder that receives the merged sections.

    Returns:
        A report of the copied photos, the skipped duplicates and the
        conflicts.
    """
    report = MergeReport()
    for batch_dir in find_batch_folders(raw_dir):
        for relative_path in list_photos(batch_dir):
            source = batch_dir / relative_path
            destination = output_dir / relative_path
            if not destination.exists():
                copy_photo(source, destination)
                report.copied.append(relative_path)
            elif filecmp.cmp(source, destination, shallow=False):
                report.already_present.append(relative_path)
            else:
                report.conflicts.append(
                    Conflict(relative_path, kept=destination, skipped=source))
    return report


print("✓ Configuration and helpers loaded (paths, photo listing, merge rule).")

✓ Configuration and helpers loaded (paths, photo listing, merge rule).


**2. Batch Inventory**

This cell takes `RAW_DIR` and, without changing anything, reports for each batch the number of photos in each section. It then counts:

1. the stone folders split across batches, whose photos are spread over more than one batch. These are the folders the merge completes
2. the distinct photo paths across all batches, which is the number of photos the output must hold

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Batch inventory  —  what each batch holds before the merge
#
# Read-only pass over RAW_DIR. A stone folder is "split" when its photos are
# spread over more than one batch; a relative path present in two batches is
# a duplicate, which the merge copies once. The number of distinct relative
# paths is therefore the exact size the merged output must reach.
# ─────────────────────────────────────────────────────────────────────────────
if "merge_batches" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")


def count_by_section(relative_paths: list[Path]) -> dict[str, int]:
    """Count photos per cemetery section.

    Args:
        relative_paths: photo paths relative to a batch.

    Returns:
        Photo counts keyed by section (the first folder of each path),
        sorted by section name.
    """
    counts: dict[str, int] = {}
    for relative_path in relative_paths:
        section = relative_path.parts[0]
        counts[section] = counts.get(section, 0) + 1
    return dict(sorted(counts.items()))


def find_split_stones(photos_by_batch: dict[Path, list[Path]]) -> set[Path]:
    """Find the stone folders whose photos are spread over several batches.

    Args:
        photos_by_batch: each batch folder mapped to its relative photo paths.

    Returns:
        The relative stone folders, e.g. `Women/Row 1/Stone 2`, that appear
        in more than one batch.
    """
    batches_per_stone: dict[Path, int] = {}
    for photos in photos_by_batch.values():
        for stone in {path.parent for path in photos}:
            batches_per_stone[stone] = batches_per_stone.get(stone, 0) + 1
    return {stone for stone, count in batches_per_stone.items() if count > 1}


batch_photos = {batch: list_photos(batch) for batch in find_batch_folders(RAW_DIR)}
all_stones = {path.parent for photos in batch_photos.values() for path in photos}
split_stones = find_split_stones(batch_photos)
expected_paths = {path for photos in batch_photos.values() for path in photos}

print(f"{RULE}\nBatch inventory: {RAW_DIR}\n{RULE}")
for _batch, _photos in batch_photos.items():
    _sections = "   ".join(f"{name}: {count:>4}"
                           for name, count in count_by_section(_photos).items())
    print(f"  {_batch.name:<26} {len(_photos):>5} photos   {_sections}")
print(f"{RULE}\n  stone folders split across batches: {len(split_stones)} of {len(all_stones)}")
print(f"  distinct photo paths (expected output size): {len(expected_paths)}")

────────────────────────────────────────────────────────────────────────
Batch inventory: data/raw
────────────────────────────────────────────────────────────────────────
  Numero 1 de Novibuco         838 photos   Children:   53   Women:  785
  Numero 1 de Novibuco 2       859 photos   Children:  306   Women:  553
  Numero 1 de Novibuco 3       931 photos   Children:   30   Women:  901
────────────────────────────────────────────────────────────────────────
  stone folders split across batches: 115 of 794
  distinct photo paths (expected output size): 2628


**3. Merge**

This cell runs `merge_batches` from `RAW_DIR` into `OUTPUT_DIR` and reports the number of photos copied, skipped as already present and left out as conflicts, listing each conflict with the file kept and the file skipped. The first run copies about 6 GB and takes a few minutes, while a re-run only compares files and copies nothing new.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Merge  —  copy every photo into OUTPUT_DIR/<section>/Row N/Stone M/
#
# The merge rule is defined in section 1. A conflict never stops the merge:
# the first batch's file stays in the output and the other is listed below
# for a person to compare by hand. The raw batches are never modified.
# ─────────────────────────────────────────────────────────────────────────────
if "merge_batches" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")

print(f"Merging {len(find_batch_folders(RAW_DIR))} batches from {RAW_DIR} into {OUTPUT_DIR}...")
merge_report = merge_batches(RAW_DIR, OUTPUT_DIR)

print(f"\n{RULE}\nMerge report\n{RULE}")
print(f"  copied:            {len(merge_report.copied):>5}")
print(f"  already present:   {len(merge_report.already_present):>5}"
      "   (identical file already in the output)")
print(f"  conflicts:         {len(merge_report.conflicts):>5}"
      "   (different files at one path, first batch kept)")
for _conflict in merge_report.conflicts[:MAX_LISTED]:
    print(f"  !! {_conflict.relative_path}\n"
          f"        kept:    {_conflict.kept}\n"
          f"        skipped: {_conflict.skipped}")
if len(merge_report.conflicts) > MAX_LISTED:
    print(f"  ... and {len(merge_report.conflicts) - MAX_LISTED} more")
print(RULE)

Merging 3 batches from data/raw into data...



────────────────────────────────────────────────────────────────────────
Merge report
────────────────────────────────────────────────────────────────────────
  copied:                0
  already present:    2628   (identical file already in the output)
  conflicts:             0   (different files at one path, first batch kept)
────────────────────────────────────────────────────────────────────────


**4. Consistency Check**

This cell rebuilds the expected output from `RAW_DIR` and compares it with each section folder of `OUTPUT_DIR`. For each section it checks that:

1. every photo of the section is present in the output
2. each output photo has the size of the file the merge kept, the first batch's copy
3. the section folder holds no stray files that come from no batch

It reports the photos, rows and stones of each section and ends with the total against the number of distinct photo paths.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Consistency check  —  the merged output against the raw batches
#
# Sizes are compared rather than bytes: the merge already compared bytes
# wherever two files met, so this pass only has to catch missing, truncated
# or stray files, and it stays fast on ~6 GB of photos. The expected output
# is rebuilt from RAW_DIR here, independently of the merge report.
# ─────────────────────────────────────────────────────────────────────────────
if "merge_batches" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")


def kept_sources(raw_dir: Path) -> dict[Path, Path]:
    """Map every distinct relative path to the batch file the merge keeps.

    Args:
        raw_dir: the folder holding one subfolder per download batch.

    Returns:
        Each relative photo path mapped to its file in the first batch, in
        name order, that holds it.
    """
    sources: dict[Path, Path] = {}
    for batch_dir in find_batch_folders(raw_dir):
        for relative_path in list_photos(batch_dir):
            sources.setdefault(relative_path, batch_dir / relative_path)
    return sources


def find_section_problems(section: str, sources: dict[Path, Path],
                          output_dir: Path) -> tuple[list[Path], list[str]]:
    """Compare one section folder of the output with the raw batches.

    Args:
        section: the section name, e.g. `Women`.
        sources: the result of `kept_sources`.
        output_dir: the folder holding the merged sections.

    Returns:
        The photos found in the section folder (relative to `output_dir`),
        and one message per missing, resized or stray photo.
    """
    expected = {path: source for path, source in sources.items()
                if path.parts[0] == section}
    section_dir = output_dir / section
    found = ([Path(section) / path for path in list_photos(section_dir)]
             if section_dir.is_dir() else [])
    problems = [f"missing  {path}" for path in expected if not (output_dir / path).is_file()]
    problems += [f"resized  {path}" for path in found if path in expected
                 and (output_dir / path).stat().st_size != expected[path].stat().st_size]
    problems += [f"stray    {path}" for path in found if path not in expected]
    return found, problems


def check_output(raw_dir: Path, output_dir: Path) -> bool:
    """Print a per-section consistency report of the merged output.

    Args:
        raw_dir: the folder holding one subfolder per download batch.
        output_dir: the folder holding the merged sections.

    Returns:
        True when every section holds exactly the expected photos.
    """
    sources = kept_sources(raw_dir)
    sections = sorted({path.parts[0] for path in sources})
    print(f"{RULE}\nConsistency: {output_dir}/<section> vs {raw_dir}\n{RULE}")
    total, failed = 0, 0
    for section in sections:
        found, problems = find_section_problems(section, sources, output_dir)
        total += len(found)
        failed += 1 if problems else 0
        rows = len({path.parts[1] for path in found})
        stones = len({path.parent for path in found})
        print(f"  {'!! ' if problems else 'OK '}{section:<10} {len(found):>5} photos"
              f" in {rows:>3} rows and {stones:>4} stones"
              + (f"  | {len(problems)} problem(s)" if problems else ""))
        for message in problems[:MAX_LISTED]:
            print(f"        - {message}")
    print(f"{RULE}\n  {total} of {len(sources)} photos in the output"
          f"  ({'consistent' if failed == 0 else str(failed) + ' section(s) need attention'})")
    return failed == 0


output_consistent = check_output(RAW_DIR, OUTPUT_DIR)

print("\n✓ Merge complete (batches merged by relative path, no file overwritten).")
print(f"  output consistent: {output_consistent}")

────────────────────────────────────────────────────────────────────────
Consistency: data/<section> vs data/raw
────────────────────────────────────────────────────────────────────────
  OK Children     389 photos in  19 rows and  195 stones


  OK Women       2239 photos in  43 rows and  599 stones
────────────────────────────────────────────────────────────────────────
  2628 of 2628 photos in the output  (consistent)

✓ Merge complete (batches merged by relative path, no file overwritten).
  output consistent: True
